# Serve Planner + Refiner LoRA Endpoints (Colab) and Compare Outputs

This notebook will:
1. Load base model `google/gemma-4-e2b-it`
2. Load both LoRA adapters (planner + refiner)
3. Expose **two ngrok endpoints**:
   - Planner endpoint
   - Refiner endpoint
4. Run sample comparisons on your paired examples and save outputs.


In [1]:
# 1) Install deps
!pip -q install -U transformers peft accelerate bitsandbytes flask pyngrok requests pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 168.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 48.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 10.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
# 2) Clone/pull repo
REPO_URL = "https://github.com/theextraordinary/VASP.git"
REPO_DIR = "/content/VASP"
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git reset --hard origin/main
%cd {REPO_DIR}


Cloning into '/content/VASP'...
remote: Enumerating objects: 470, done.
remote: Counting objects: 100% (470/470), done.
remote: Compressing objects: 100% (337/337), done.
remote: Total 470 (delta 129), reused 465 (delta 125), pack-reused 0 (from 0)
Receiving objects: 100% (470/470), 6.48 MiB | 43.34 MiB/s, done.
Resolving deltas: 100% (129/129), done.
/content/VASP


In [3]:
# 3) Mount Drive and set adapter zip paths
from google.colab import drive
drive.mount('/content/drive')

PLANNER_ZIP = "/content/drive/MyDrive/planner_e2b_qlora.zip"
REFINER_ZIP = "/content/drive/MyDrive/refiner_e2b_qlora.zip"

!test -f "$PLANNER_ZIP" && echo "Found planner zip" || echo "Missing planner zip"
!test -f "$REFINER_ZIP" && echo "Found refiner zip" || echo "Missing refiner zip"


Mounted at /content/drive
Found planner zip
Found refiner zip


In [4]:
# 4) Unzip adapters
!mkdir -p /content/adapters/planner /content/adapters/refiner
!unzip -o "$PLANNER_ZIP" -d /content/adapters/planner >/dev/null
!unzip -o "$REFINER_ZIP" -d /content/adapters/refiner >/dev/null

# Auto-detect inner adapter dirs (containing adapter_config.json)
import os
from pathlib import Path

def find_adapter_dir(root):
    root = Path(root)
    for p in root.rglob('adapter_config.json'):
        return str(p.parent)
    return None

planner_adapter_dir = find_adapter_dir('/content/adapters/planner')
refiner_adapter_dir = find_adapter_dir('/content/adapters/refiner')
print('planner_adapter_dir=', planner_adapter_dir)
print('refiner_adapter_dir=', refiner_adapter_dir)
assert planner_adapter_dir and refiner_adapter_dir, 'Could not locate adapter_config.json in one of the zips'


planner_adapter_dir= /content/adapters/planner/finetune/output/planner_e2b_qlora
refiner_adapter_dir= /content/adapters/refiner/finetune/output/refiner_e2b_qlora


In [5]:
# 5) Load base model + both adapters (single model, switch adapter per request)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "google/gemma-4-e2b-it"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base, planner_adapter_dir, adapter_name='planner')
model.load_adapter(refiner_adapter_dir, adapter_name='refiner')
model.eval()
print('Adapters loaded:', model.peft_config.keys())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Adapters loaded: dict_keys(['planner', 'refiner'])


In [9]:
!pkill -f "python.*flask" || true
!pkill -f "python.*8096" || true
!pkill -f "python.*8089" || true

from pyngrok import ngrok
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
ngrok.kill()


^C
^C


^C


In [2]:
!kill -9 $(lsof -t -i:8089)

: 

: 

: 

In [6]:
# 6) Start ONE Flask app + ONE ngrok tunnel (planner/refiner/base routes)
import threading
from threading import Lock
from flask import Flask, request, jsonify
from pyngrok import ngrok
import torch

# Set your ngrok token first
NGROK_AUTHTOKEN = "3CGpBa50gsAphlT8CJAcg1mJJ49_4W992pra1ULgrNqrCmYTf"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Clean old tunnels if any
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass

_gen_lock = Lock()

def _generate_text(prompt: str, max_new_tokens: int, temperature: float) -> str:
    messages = [{"role": "user", "content": prompt}]
    enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(model.device)

    gen_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=max(temperature, 1e-5),
        top_p=0.95,
        repetition_penalty=1.10,
        pad_token_id=tokenizer.eos_token_id,
    )

    with torch.no_grad():
        out = model.generate(**gen_kwargs)

    gen_ids = out[0][input_ids.shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def run_generate(prompt: str, mode: str, max_new_tokens: int = 900, temperature: float = 0.0):
    with _gen_lock:
        mode = (mode or 'planner').lower()
        if mode in {"planner", "refiner"}:
            model.set_adapter(mode)
            return _generate_text(prompt, max_new_tokens, temperature)

        # actual base E2B (no LoRA adapter)
        if mode in {"base", "e2b", "actual"}:
            if hasattr(model, "disable_adapter"):
                with model.disable_adapter():
                    return _generate_text(prompt, max_new_tokens, temperature)
            # fallback if disable_adapter is unavailable
            model.set_adapter("planner")
            return _generate_text(prompt, max_new_tokens, temperature)

        raise ValueError(f"Unknown mode: {mode}")


app = Flask("planner_refiner_app")


@app.route("/planner/generate", methods=["POST"])
def planner_generate():
    data = request.get_json(force=True)
    prompt = data.get("prompt", "")
    temperature = float(data.get("temperature", 0.0))
    max_tokens = int(data.get("max_tokens", 1200))
    text = run_generate(prompt, "planner", max_tokens, temperature)
    return jsonify({"response": text})


@app.route("/refiner/generate", methods=["POST"])
def refiner_generate():
    data = request.get_json(force=True)
    prompt = data.get("prompt", "")
    temperature = float(data.get("temperature", 0.0))
    max_tokens = int(data.get("max_tokens", 2200))
    text = run_generate(prompt, "refiner", max_tokens, temperature)
    return jsonify({"response": text})


@app.route("/base/generate", methods=["POST"])
def base_generate():
    data = request.get_json(force=True)
    prompt = data.get("prompt", "")
    temperature = float(data.get("temperature", 0.0))
    max_tokens = int(data.get("max_tokens", 1200))
    text = run_generate(prompt, "base", max_tokens, temperature)
    return jsonify({"response": text})


PORT = 8089

# Start Flask once
threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False),
    daemon=True
).start()

# Expose one ngrok tunnel
public_base = ngrok.connect(PORT, "http").public_url
planner_url = public_base + "/planner/generate"
refiner_url = public_base + "/refiner/generate"
base_url = public_base + "/base/generate"

print("Planner endpoint:", planner_url)
print("Refiner endpoint:", refiner_url)
print("Base E2B endpoint:", base_url)



 * Serving Flask app 'planner_refiner_app'                                                          
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8089
 * Running on http://172.28.0.12:8089
INFO:werkzeug:Press CTRL+C to quit


Planner endpoint: https://bungee-swell-smashing.ngrok-free.dev/planner/generate
Refiner endpoint: https://bungee-swell-smashing.ngrok-free.dev/refiner/generate
Base E2B endpoint: https://bungee-swell-smashing.ngrok-free.dev/base/generate


In [10]:
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

In [ ]:
# 7) Quick endpoint smoke test
import requests

r1 = requests.post(planner_url, json={'prompt': 'Say planner online in one line.', 'max_tokens': 32, 'temperature': 0.0}, timeout=120)
r2 = requests.post(refiner_url, json={'prompt': 'Return {"ok":true} only.', 'max_tokens': 32, 'temperature': 0.0}, timeout=120)
r3 = requests.post(base_url, json={'prompt': 'Say base e2b online in one line.', 'max_tokens': 32, 'temperature': 0.0}, timeout=120)
print('planner:', r1.status_code, r1.json().get('response','')[:120])
print('refiner:', r2.status_code, r2.json().get('response','')[:120])
print('base   :', r3.status_code, r3.json().get('response','')[:120])



[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 21:30:13] "POST /planner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 21:30:13] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 21:30:14] "POST /base/generate HTTP/1.1" 200 -


planner: 200 **Planner online allows you to manage your schedule and tasks digitally.**
refiner: 200 {"ok":true}
base   : 200 Here is the phrase "base e2b online" in one line:

**Base E2B online**


INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 21:35:18] "POST /planner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 21:35:32] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 21:49:52] "POST /planner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 21:51:10] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 22:04:53] "POST /planner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 22:05:22] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 22:05:53] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 22:06:28] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 22:06:30] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 22:06:58] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 22:07:21] "POST /refiner/generate HTTP/1.1" 200 -

In [8]:
# 8) Load paired examples (planner + refiner) and run comparison
import json, random, re
from pathlib import Path
import requests

planner_rows = [json.loads(l) for l in Path('finetune/data/planner_train_paired.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
refiner_rows = [json.loads(l) for l in Path('finetune/data/refiner_train_paired.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]

planner_map = {r['id'].replace('planner_',''): r for r in planner_rows}
refiner_map = {r['id'].replace('refiner_',''): r for r in refiner_rows}
common_ids = sorted(set(planner_map) & set(refiner_map))
print('paired ids:', len(common_ids))

random.seed(42)
sample_ids = random.sample(common_ids, 5)
print('sample ids:', sample_ids)

def msg_text(messages, role):
    for m in messages:
        if m.get('role') == role:
            return m.get('content','')
    return ''

results = []
for sid in sample_ids:
    prow = planner_map[sid]
    rrow = refiner_map[sid]

    p_user = msg_text(prow['messages'], 'user')
    p_gold = msg_text(prow['messages'], 'assistant')

    planner_resp = requests.post(planner_url, json={
        'prompt': p_user,
        'temperature': 0.0,
        'max_tokens': 1400,
    }, timeout=300).json().get('response','')

    r_user = msg_text(rrow['messages'], 'user')
    r_gold = msg_text(rrow['messages'], 'assistant')

    refiner_resp = requests.post(refiner_url, json={
        'prompt': r_user,
        'temperature': 0.0,
        'max_tokens': 2600,
    }, timeout=300).json().get('response','')

    # Basic checks
    planner_ok = all(h in planner_resp for h in ['EDIT PLAN', 'Global Style:', 'Segment 1'])
    try:
        parsed = json.loads(refiner_resp)
        refiner_json_ok = isinstance(parsed, dict) and 'elements' in parsed
    except Exception:
        refiner_json_ok = False

    results.append({
        'id': sid,
        'planner_ok': planner_ok,
        'refiner_json_ok': refiner_json_ok,
        'planner_pred': planner_resp,
        'planner_gold': p_gold,
        'refiner_pred': refiner_resp,
        'refiner_gold': r_gold,
    })

out = Path('output')
out.mkdir(exist_ok=True)
Path('output/ft_planner_refiner_compare.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved: output/ft_planner_refiner_compare.json')
print('planner_ok:', sum(1 for r in results if r['planner_ok']), '/', len(results))
print('refiner_json_ok:', sum(1 for r in results if r['refiner_json_ok']), '/', len(results))


paired ids: 420
sample ids: ['image_anim_design_068', 'caption_timing_068', 'caption_layout_103', 'person_anim_20', 'edit_pattern1_031']


INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:26:53] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:27:25] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:27:36] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:28:13] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:28:24] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:28:59] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:29:23] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:29:37] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:30:23] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 15:31:00] "POST /generate HTTP/1.1" 200 -


Saved: output/ft_planner_refiner_compare.json
planner_ok: 2 / 5
refiner_json_ok: 1 / 5


In [ ]:
# 9) Pretty print a short comparison report
import json
from pathlib import Path

rows = json.loads(Path('output/ft_planner_refiner_compare.json').read_text(encoding='utf-8'))
for r in rows:
    print('='*100)
    print('ID:', r['id'])
    print('Planner format ok:', r['planner_ok'])
    print('Refiner JSON ok:', r['refiner_json_ok'])
    print('
Planner prediction (head):
', r['planner_pred'][:500].replace('
',' '))
    print('
Refiner prediction (head):
', r['refiner_pred'][:500].replace('
',' '))


In [ ]:
# 10) Optional: set env vars and run your pipeline against served endpoints
# (planner/refiner pipeline uses tuned endpoints; base endpoint is for direct comparison only)
import os
os.environ['E2B_URL'] = planner_url.rsplit('/generate',1)[0]
os.environ['E4B_URL'] = refiner_url.rsplit('/generate',1)[0]
os.environ['BASE_E2B_URL'] = base_url.rsplit('/generate',1)[0]
print('E2B_URL=', os.environ['E2B_URL'])
print('E4B_URL=', os.environ['E4B_URL'])
print('BASE_E2B_URL=', os.environ['BASE_E2B_URL'])

# Example run:
# !python -m vasp.a2v.main --use-dummy --mode planner_to_video

